In [29]:
import requests
import csv

# Elprisanalys
Detta projekt hämtar elpriser från API:et "Elpriset just nu", analyserar 
dyraste/billigaste tider och sparar resultatet i en CSV-fil.

## Hämta priser från API
Funktionen hämtar elpriser för ett visst datum och elområde från "Elpriset just nu". 
Om anropet misslyckas (t.ex. fel datum eller nätverksfel) fångas felet med try/except 
så att programmet inte kraschar.

In [30]:
def hämta_priser(datum, elområde):
    url = f"https://www.elprisetjustnu.se/api/v1/prices/{datum}_{elområde}.json"
    try:
        svar = requests.get(url)
        data = svar.json()
        return data
    except Exception as e:
        print("Något gick fel vid hämtning av priser.")
        print("Riktiga felet:", e)
        return []

## Skapa prisobservationer
Funktionen går igenom all rådata från API:et och skapar ett objekt för varje tidpunkt. 
Den tidpunkt som har dygnets högsta pris får ett ToppPrisObservation-objekt istället, 
för att visa att arv faktiskt används i praktiken.

## Analysera priser
Funktionen går igenom listan av prisobservationer och skriver ut högsta, lägsta 
och genomsnittligt pris. Den letar också upp eventuella ToppPrisObservation-objekt 
och visar dem separat.

In [31]:
def skapa_prisobservationer(data):
    observationer = []
    högsta_pris = max(post["SEK_per_kWh"] for post in data)

    for post in data:
        tid = post["time_start"]
        pris = post["SEK_per_kWh"]

        if pris == högsta_pris:
            observation = ToppPrisObservation(tid, pris, "Dygnets högsta pris")
        else:
            observation = Prisobservation(tid, pris)

        observationer.append(observation)

    return observationer

In [32]:
def analysera_priser(observationer):
    priser = [obs.pris for obs in observationer]
    print("Högsta pris:", max(priser), "SEK/kWh")
    print("Lägsta pris:", min(priser), "SEK/kWh")
    print("Medelpris:", sum(priser) / len(priser), "SEK/kWh")

    for obs in observationer:
        if isinstance(obs, ToppPrisObservation):
            print("Toppobservation:", obs.tid, "-", obs.anledning)

## Spara till CSV och kör hela programmet
Den här delen sparar resultatet till en CSV-fil och kör sedan hela programmet från 
början till slut: hämtar data, skapar objekt, analyserar och sparar.

In [33]:
class Prisobservation:
    def __init__(self, tid, pris):
        self.tid = tid
        self.pris = pris

    def är_dyrt(self):
        return self.pris > 1.50


class ToppPrisObservation(Prisobservation):
    def __init__(self, tid, pris, anledning):
        super().__init__(tid, pris)
        self.anledning = anledning

In [34]:
def spara_till_csv(observationer, filnamn):
    with open(filnamn, "w", newline="", encoding="utf-8") as fil:
        writer = csv.writer(fil)
        writer.writerow(["tid", "pris_sek_per_kwh", "är_dyrt"])
        for obs in observationer:
            writer.writerow([obs.tid, obs.pris, obs.är_dyrt()])
    print("Datan sparades i", filnamn)

## Spara till CSV och kör hela programmet
Den här delen sparar resultatet till en CSV-fil och kör sedan hela programmet från 
början till slut: hämtar data, skapar objekt, analyserar och sparar.

In [ ]:
def main():
    datum = "2026/09-24"
    elområde = "SE3"

    rådata = hämta_priser(datum, elområde)

    if rådata:
        observationer = skapa_prisobservationer(rådata)
        analysera_priser(observationer)
        spara_till_csv(observationer, "elpriser.csv")
    else:
        print("Ingen data kunde hämtas.")

main() 

Högsta pris: 2.56001 SEK/kWh
Lägsta pris: 1.09848 SEK/kWh
Medelpris: 1.6284937499999999 SEK/kWh
Toppobservation: 2026-09-24T18:45:00+02:00 - Dygnets högsta pris
Datan sparades i elpriser.csv


## Klasser: Prisobservation och ToppPrisObservation
Varje prisobservation representeras av ett objekt. Om en observation råkar vara 
dygnets högsta pris skapas istället ett objekt av barnklassen ToppPrisObservation, 
som ärver från Prisobservation men även sparar en anledning.